# Arabic Wav2Vec2 -> OpenVINO CPU-Produktionsbackend

Dieses Notebook exportiert das Modell einmal nach OpenVINO IR und startet danach ausschließlich den optimierten CPU-Backendpfad für die Mobile-App.

Produktionsprinzipien: kein GPU-Pfad, kein PyTorch-Referenzmodell, keine Benchmark-/Testaudio-Zellen, ein kompiliertes OpenVINO-Modell mit Latenzprofil, begrenztem Einzelstream und persistentem Modellcache. `torch` bleibt nur für `forced_align` und die vorhandene VAD-/Scoring-Pipeline geladen.

## 1. Installation

Nur Pakete installieren, die der Export oder der produktive CPU-Backendpfad benötigt. Nach einer neuen Installation den Kernel einmal neu starten.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
import sys

PIP_PACKAGES = [
    "optimum-intel[openvino]", "transformers", "torch", "torchaudio",
    "pydub", "scipy", "python-multipart", "fastapi", "uvicorn[standard]", "silero-vad",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)

if shutil.which("ffmpeg") is None:
    if os.path.exists("/content") and shutil.which("apt-get"):
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
    else:
        raise RuntimeError("ffmpeg fehlt. Bitte ffmpeg installieren und die Zelle erneut ausführen.")

for mod in ("transformers", "openvino", "torchaudio", "fastapi", "uvicorn", "silero_vad"):
    print(f"{mod:14s} {'OK' if importlib.util.find_spec(mod) else 'FEHLT'}")
print("CPU-Produktionsabhängigkeiten bereit.")

Installierte Kernpakete:
  transformers   OK
  openvino       OK
  nncf           OK
  fastapi        OK
  uvicorn        OK
  silero_vad     OK
✅ Abhaengigkeiten bereit. Kein automatischer Kernel-Neustart erforderlich.


In [ ]:
import openvino as ov
from optimum.intel import OVModelForCTC
from transformers import Wav2Vec2Processor

MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
OV_FP32_DIR = "ov_wav2vec2_ar_fp32"

processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
ov_model = OVModelForCTC.from_pretrained(MODEL_ID, export=True, compile=False)
ov_model.save_pretrained(OV_FP32_DIR)
processor.save_pretrained(OV_FP32_DIR)

print(f"OpenVINO IR exportiert: {OV_FP32_DIR}")
print(f"CPU: {ov.Core().get_property('CPU', 'FULL_DEVICE_NAME')}")

---

# Produktionsbackend

Ab hier läuft ausschließlich der OpenVINO-CPU-Forward. `torch` wird nur für Silero-VAD und `forced_align` verwendet.

Ausführungsreihenfolge: Backend-Imports, OpenVINO-Modell/VAD laden, Preprocessing/Scoring definieren, FastAPI-App definieren, Funnel starten.

In [ ]:
import io, os, time, unicodedata
from types import SimpleNamespace
from typing import List, Dict, Any

import numpy as np
import torch
import torchaudio
import torchaudio.functional as AF
from pydub import AudioSegment
from silero_vad import load_silero_vad, get_speech_timestamps

SR = 16000
device = torch.device("cpu")
print(f"CPU-Backend: torch {torch.__version__} (nur VAD/forced_align), OpenVINO-Forward")

## OpenVINO-CPU-Modell laden

Der ASR-Forward wird durch `OVCTCModel` auf OpenVINO ersetzt. Der aktive Pfad nutzt
bewusst das FP32-IR aus Schritt 3: auf einer CPU ist FP16 nicht automatisch schneller
und kann je nach Hardware intern wieder in FP32 umgewandelt werden.

Ein `threading.Lock` serialisiert die wiederverwendete `InferRequest`; das passt zum
Einzel-Request-Betrieb des Backends und verhindert parallele Zugriffe auf denselben
OpenVINO-Request.

In [ ]:
import threading
import openvino as ov
from transformers import Wav2Vec2Processor

MODEL_DIR = OV_FP32_DIR
assert os.path.exists(f"{MODEL_DIR}/openvino_model.xml"), (
    f"{MODEL_DIR} fehlt - zuerst den Export ausführen.")

ASR_MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic (OpenVINO CPU FP32)"
DTYPE = torch.float32

# Für einzelne, interaktive Requests: minimale Scheduling-Latenz und kein Thread-Overhead.
N_THREADS = int(os.environ.get("OV_THREADS", max(1, min(8, os.cpu_count() or 1))))
N_THREADS = max(1, min(N_THREADS, os.cpu_count() or 1))
OV_RUNTIME_CONFIG = {
    "PERFORMANCE_HINT": "LATENCY",
    "NUM_STREAMS": "1",
    "INFERENCE_NUM_THREADS": N_THREADS,
    "CACHE_DIR": os.environ.get("OV_CACHE_DIR", "ov_cache"),
}

class OVCTCModel:
    """Wiederverwendbarer, thread-sicherer OpenVINO-Request für den CTC-Forward."""

    def __init__(self, model_dir: str, cfg: dict):
        core = ov.Core()
        self.compiled = core.compile_model(f"{model_dir}/openvino_model.xml", "CPU", cfg)
        self.req = self.compiled.create_infer_request()
        self._in = self.compiled.input(0)
        self._out = self.compiled.output(0)
        self._lock = threading.Lock()
        self.config = SimpleNamespace(pad_token_id=None)

    def __call__(self, input_values):
        x = input_values.detach().cpu().numpy().astype(np.float32, copy=False)
        with self._lock:
            logits = self.req.infer({self._in: x})[self._out].copy()
        return SimpleNamespace(logits=torch.from_numpy(logits))

    def eval(self):
        return self

print(f"Lade OpenVINO CPU-Modell ({N_THREADS} Threads) …")
asr_processor = Wav2Vec2Processor.from_pretrained(MODEL_DIR)
asr_model = OVCTCModel(MODEL_DIR, OV_RUNTIME_CONFIG)
ASR_VOCAB = asr_processor.tokenizer.get_vocab()
ASR_BLANK_ID = asr_processor.tokenizer.pad_token_id
asr_model.config.pad_token_id = ASR_BLANK_ID

print("Lade Silero VAD …")
vad_model = load_silero_vad()

# Ein Warm-up vermeidet Modell-/Shape-Kompilierung bei der ersten Nutzeranfrage.
_t0 = time.perf_counter()
asr_model(torch.zeros(1, 2 * SR)).logits
print(f"CPU bereit, Warm-up {(time.perf_counter() - _t0) * 1e3:.0f} ms")

## Schritt 12 — Audio-Preprocessing und Scoring

Beide Zellen wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zellen 4 und 5).
Die Signalkette (`decode → HP 80 Hz → RMS-Norm → gentle_trim → Kontext-Pad`), das
GOP-/LLR-Scoring, `forced_align` und die Tajweed-Schwellen sind unverändert — nur
so sind die Ergebnisse mit dem GPU-Backend vergleichbar.

`run_asr` in der zweiten Zelle ruft `asr_model(...)` auf und trifft damit
automatisch den OpenVINO-Shim aus Schritt 11. **`forced_align` bleibt torch** —
nur der Modell-Forward wandert zu OpenVINO.

In [ ]:
from scipy.signal import butter, sosfiltfilt

_HPF_SOS = butter(2, 80.0, btype="highpass", fs=SR, output="sos")

def decode_audio(raw: bytes) -> np.ndarray:
    """Schneller nativer Decoder; ffmpeg/pydub bleibt nur als Format-Fallback."""
    try:
        wav, sample_rate = torchaudio.load(io.BytesIO(raw))
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sample_rate != SR:
            wav = AF.resample(wav, sample_rate, SR)
        return wav.squeeze(0).contiguous().numpy().astype(np.float32, copy=False)
    except Exception:
        seg = AudioSegment.from_file(io.BytesIO(raw))
        seg = seg.set_frame_rate(SR).set_channels(1).set_sample_width(2)
        return np.asarray(seg.get_array_of_samples(), dtype=np.float32) / 32768.0

def _highpass(audio: np.ndarray) -> np.ndarray:
    return sosfiltfilt(_HPF_SOS, audio).astype(np.float32)

def _normalize_level(audio: np.ndarray, target_dbfs: float = -20.0) -> np.ndarray:
    rms = float(np.sqrt(np.mean(audio ** 2)))
    if rms < 1e-6:
        return audio
    gain = 10.0 ** ((target_dbfs - 20.0 * np.log10(rms)) / 20.0)
    out = audio * gain
    peak = float(np.max(np.abs(out)))
    if peak > 0.99:
        out = out / peak * 0.99
    return out.astype(np.float32)

def gentle_trim(audio: np.ndarray, pad_ms: int = 120) -> np.ndarray:
    segs = get_speech_timestamps(torch.from_numpy(audio), vad_model, sampling_rate=SR, threshold=0.35)
    if not segs:
        return audio
    pad = int(pad_ms * SR / 1000)
    return audio[max(0, segs[0]["start"] - pad):min(len(audio), segs[-1]["end"] + pad)]

def _pad_context(audio: np.ndarray, ms: int = 250) -> np.ndarray:
    pad = np.zeros(int(ms * SR / 1000), dtype=np.float32)
    return np.concatenate([pad, audio, pad])

def preprocess(raw: bytes) -> np.ndarray:
    audio = _normalize_level(_highpass(decode_audio(raw)))
    return _pad_context(gentle_trim(audio))

In [ ]:
# Nur klassisches Tashkeel entfernen. Hamza-Formen (أ إ آ ؤ ئ) bleiben als eigene Buchstaben erhalten.
_TASHKEEL = set("ًٌٍَُِّْٰ")

def strip_diacritics(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if c not in _TASHKEEL))

# Positionsabhaengige Aequivalenzen (Anfang/Ende) fuer Posterior-Bewertung.
_START_EQUIV = {ch: "اأإآ" for ch in "اأإآ"}
_END_EQUIV   = {"ة": "ةه", "ه": "هة",
                "ى": "ىيا", "ي": "يى"}

# Linguistisch belegte Verwechslungen fuer den LLR-Test.
# Quellen: Al-Ani (1970) "Arabic Phonology"; Newman (2013);
# Standard-DaF/L2-Arabisch-Fehlerkataloge; Kinder-L1-Erwerbsstudien.
_CONFUSABLES: Dict[str, str] = {
    "ت": "طثد",
    "ث": "تسذف",
    "ح": "هخع",
    "خ": "حغك",
    "د": "تضذ",
    "ذ": "دزثظ",
    "ر": "لغ",
    "ز": "ذسظ",
    "س": "صثزش",
    "ش": "سج",
    "ص": "سض",
    "ض": "دظص",
    "ط": "تضد",
    "ظ": "زذض",
    "ع": "ءأاه",
    "غ": "خقر",
    "ق": "كغخ",
    "ك": "قخج",
    "ل": "ر",
    "ه": "حة",
    "ء": "ع",
    "ج": "شك",
}

def _equiv_ids(ch: str, pos: int, total: int) -> List[int]:
    if pos == 0 and ch in _START_EQUIV:
        alts = _START_EQUIV[ch]
    elif pos == total - 1 and ch in _END_EQUIV:
        alts = _END_EQUIV[ch]
    else:
        alts = ch
    ids = [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]
    return ids or [ASR_VOCAB[ch]]

def _confuse_ids(ch: str) -> List[int]:
    alts = _CONFUSABLES.get(ch, "")
    return [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]

# Umkehr-Map: Token-ID -> Buchstabe, fuer error_hint.
_ID_TO_CHAR = {tid: c for c, tid in ASR_VOCAB.items()}

def encode_target(word: str) -> List[int]:
    ids: List[int] = []
    for ch in word:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise ValueError(f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        ids.append(tid)
    return ids

@torch.inference_mode()
def run_asr(audio: np.ndarray):
    inputs = asr_processor(audio, sampling_rate=SR, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device=device, dtype=DTYPE)
    logits = asr_model(input_values).logits
    # log_softmax stabil in fp32, torchaudio.forced_align verlangt float32 CPU.
    log_probs = torch.log_softmax(logits.float(), dim=-1).cpu()
    transcription = asr_processor.batch_decode(log_probs.argmax(dim=-1))[0]
    return log_probs, transcription

def _runs_of_non_blank(tokens: List[int]) -> List[List[int]]:
    runs: List[List[int]] = []
    current: List[int] = []
    last: int = -1
    for t, tok in enumerate(tokens):
        if tok == ASR_BLANK_ID:
            if current: runs.append(current); current = []
            last = -1
        elif tok != last:
            if current: runs.append(current)
            current = [t]; last = tok
        else:
            current.append(t)
    if current: runs.append(current)
    return runs

# Kalibrierungskonstante: LLR=0 -> 50, LLR=+1 -> ~88, LLR=-1 -> ~12.
_LLR_K = 2.0

def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + float(np.exp(-x)))

def gop_score(log_probs: torch.Tensor, target_word: str) -> List[Dict[str, Any]]:
    target_ids = encode_target(target_word)
    if not target_ids:
        return []
    if log_probs.shape[1] < len(target_ids):
        raise ValueError("Aufnahme zu kurz für dieses Wort.")
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    total_len = len(target_word)
    results: List[Dict[str, Any]] = []
    for i, ch in enumerate(target_word):
        if i >= len(runs):
            results.append({"label": ch, "score": 0.0, "confidence": 0.0,
                            "llr": -5.0, "error_hint": None})
            continue

        frames   = runs[i]
        lp_frame = log_probs[0, frames]  # [F, V]

        # 1) Posterior-Score (klassisches GOP, positionsbewusst).
        equiv_ids  = _equiv_ids(ch, i, total_len)
        target_lp  = lp_frame[:, equiv_ids].max(dim=-1).values.mean().item()
        post_score = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
        conf       = float(np.exp(target_lp))

        # 2) LLR gegen dokumentierte Verwechslungen (Anti-Modell).
        confuse_ids = _confuse_ids(ch)
        if confuse_ids:
            per_frame_conf = lp_frame[:, confuse_ids]
            best_conf_lp   = per_frame_conf.max(dim=-1).values.mean().item()
            llr            = target_lp - best_conf_lp
            llr_score      = _sigmoid(_LLR_K * llr) * 100.0
            # Nur melden wenn Verwechslung staerker als Ziel.
            if llr < 0:
                best_col   = int(per_frame_conf.mean(dim=0).argmax().item())
                hint_id    = confuse_ids[best_col]
                error_hint = _ID_TO_CHAR.get(hint_id)
            else:
                error_hint = None
        else:
            llr, llr_score, error_hint = 5.0, 100.0, None

        # 3) Kombination: 40 % Posterior + 60 % LLR (LLR ist informativer).
        final = 0.4 * post_score + 0.6 * llr_score
        results.append({
            "label": ch,
            "score": float(np.clip(final, 0, 100)),
            "confidence": conf,
            "llr": float(llr),
            "error_hint": error_hint,
        })
    return results

## Schritt 13 — FastAPI-App

Wörtlich aus `wav2vec2_arabic_pronunciation.ipynb` (Zelle 6): `/assess` (HTTP),
`/stream` (WebSocket, Wort- und Ayah-Modus), `/health`, `/logs`. Auth per
`API_TOKEN` — aus dem Colab-Secret `API_TOKEN`, sonst automatisch generiert und
in Schritt 14 ausgegeben.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, Iterator
from collections import deque
from datetime import datetime, timezone

MAX_AUDIO_BYTES = 3 * 1024 * 1024
MAX_AYAH_AUDIO_BYTES = 8 * 1024 * 1024
MIN_SAMPLES = int(0.15 * SR)
WS_KEEPALIVE_SEC = 20
WORD_STREAM_DELAY_SEC = 0.035

_LOG_BUF: deque = deque(maxlen=200)

def _log(event: str, **kv):
    entry = {"ts": datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z"), "event": event, **kv}
    _LOG_BUF.append(entry)
    print(f"[{entry['ts']}] {event}  " + " ".join(f"{k}={v}" for k, v in kv.items()), flush=True)

class Unit(BaseModel):
    label: str
    score: float
    confidence: float
    llr: Optional[float] = None
    error_hint: Optional[str] = None

class AssessResponse(BaseModel):
    target: str
    transcription: str
    units: List[Unit]
    total: float
    duration_ms: int
    timings: Optional[Dict[str, int]] = None

app = FastAPI(title="Arabic Pronunciation API", version="3.0.0-openvino-cpu")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

import secrets as _secrets
try:
    from google.colab import userdata as _ud
    API_TOKEN = _ud.get("API_TOKEN") or None
except Exception:
    API_TOKEN = None
if not API_TOKEN:
    API_TOKEN = os.environ.get("API_TOKEN") or _secrets.token_urlsafe(24)

from fastapi import Header, Query

def _require_token(header_token: Optional[str], query_token: Optional[str]) -> None:
    provided = header_token or query_token or ""
    if not provided or not _secrets.compare_digest(provided, API_TOKEN):
        raise HTTPException(401, "Ungueltiger oder fehlender API-Token.")

def _auth_dep(x_api_token: Optional[str] = Header(default=None, alias="X-API-Token"), token: Optional[str] = Query(default=None)) -> None:
    _require_token(x_api_token, token)

@app.middleware("http")
async def _request_logger(request, call_next):
    t0 = time.perf_counter()
    resp = await call_next(request)
    if not request.url.path.startswith("/logs"):
        _log("http", method=request.method, path=request.url.path, status=resp.status_code, ms=int((time.perf_counter() - t0) * 1000), client=request.client.host if request.client else "?")
    return resp

def _score_word(raw: bytes, target: str) -> Dict[str, Any]:
    if len(raw) > MAX_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")
    t_total = time.perf_counter()
    t_pre = time.perf_counter()
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungueltig: {e}")
    preprocess_ms = int((time.perf_counter() - t_pre) * 1000)
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    target_clean = strip_diacritics(target)
    t_asr = time.perf_counter()
    log_probs, transcription = run_asr(wav)
    asr_ms = int((time.perf_counter() - t_asr) * 1000)
    t_score = time.perf_counter()
    units = gop_score(log_probs, target_clean)
    score_ms = int((time.perf_counter() - t_score) * 1000)
    total_ms = int((time.perf_counter() - t_total) * 1000)
    timings = {"preprocess_ms": preprocess_ms, "asr_ms": asr_ms, "score_ms": score_ms, "audio_ms": int(wav.size * 1000 / SR), "server_ms": total_ms}
    _log("assess_compute", bytes=len(raw), target_chars=len(target_clean), **timings)
    return {"target": target_clean, "transcription": transcription, "units": units, "total": float(np.mean([u["score"] for u in units])) if units else 0.0, "timings": timings}

In [ ]:
from fastapi import Depends, File, Form, UploadFile, WebSocket, WebSocketDisconnect
import asyncio
import json

@app.get("/health")
def health():
    return {
        "status": "ok", "device": "CPU", "asr_model": ASR_MODEL_ID,
        "vad": "Silero VAD 5", "fp16": False,
        "endpoints": ["/assess (HTTP)", "/stream (WebSocket)", "/logs"],
    }

@app.get("/logs", dependencies=[Depends(_auth_dep)])
def get_logs(n: int = 50):
    n = max(1, min(int(n), _LOG_BUF.maxlen or 200))
    entries = list(_LOG_BUF)[-n:][::-1]
    return {"count": len(entries), "entries": entries}

@app.post("/assess", response_model=AssessResponse, dependencies=[Depends(_auth_dep)])
def assess(audio: UploadFile = File(...), target: str = Form(...)):
    target = target.strip()
    if not target:
        raise HTTPException(400, "Zielwort fehlt.")
    t0 = time.perf_counter()
    result = _score_word(audio.file.read(MAX_AUDIO_BYTES + 1), target)
    result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
    return AssessResponse(units=[Unit(**u) for u in result.pop("units")], **result)

def _score_ayah_streamed(raw: bytes, ayah_text: str) -> Iterator[Dict[str, Any]]:
    if len(raw) > MAX_AYAH_AUDIO_BYTES or not raw:
        raise HTTPException(413 if raw else 400, "Audio ungueltig oder zu gross.")
    words = [strip_diacritics(w) for w in ayah_text.split() if strip_diacritics(w)]
    if not words:
        raise HTTPException(400, "Ayah-Text leer.")
    target_text = "".join(words)
    target_ids = encode_target(target_text)
    t_pre = time.perf_counter()
    wav = preprocess(raw)
    pre_ms = int((time.perf_counter() - t_pre) * 1000)
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    t_asr = time.perf_counter()
    log_probs, transcription = run_asr(wav)
    asr_ms = int((time.perf_counter() - t_asr) * 1000)
    if log_probs.shape[1] < len(target_ids):
        raise HTTPException(400, "Aufnahme zu kurz fuer diese Ayah.")
    t_align = time.perf_counter()
    aligned, _ = AF.forced_align(log_probs, torch.tensor([target_ids], dtype=torch.int32), blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    align_ms = int((time.perf_counter() - t_align) * 1000)
    yield {"kind": "start", "words_count": len(words), "transcription": transcription}
    cursor = 0
    word_scores = []
    for word_idx, word in enumerate(words):
        units = []
        for local_idx, char in enumerate(word):
            frame_idx = cursor + local_idx
            if frame_idx >= len(runs):
                units.append({"label": char, "score": 0.0, "confidence": 0.0, "llr": -5.0, "error_hint": None})
                continue
            lp = log_probs[0, runs[frame_idx]]
            target_lp = lp[:, _equiv_ids(char, local_idx, len(word))].max(dim=-1).values.mean().item()
            post = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
            confuse_ids = _confuse_ids(char)
            if confuse_ids:
                conf_lp = lp[:, confuse_ids]
                llr = target_lp - conf_lp.max(dim=-1).values.mean().item()
                llr_score = _sigmoid(_LLR_K * llr) * 100.0
                hint = _ID_TO_CHAR.get(confuse_ids[int(conf_lp.mean(dim=0).argmax().item())]) if llr < 0 else None
            else:
                llr, llr_score, hint = 5.0, 100.0, None
            units.append({"label": char, "score": float(np.clip(0.4 * post + 0.6 * llr_score, 0, 100)), "confidence": float(np.exp(target_lp)), "llr": float(llr), "error_hint": hint})
        cursor += len(word)
        scores = [u["score"] for u in units]
        score = float(np.clip(0.75 * np.mean(scores) + 0.25 * np.min(scores), 0, 100)) if scores else 0.0
        word_scores.append(score)
        yield {"kind": "word", "word_idx": word_idx, "target": word, "score": score, "units": units}
    yield {"kind": "done", "total": float(np.mean(word_scores)) if word_scores else 0.0, "words_count": len(words), "timings": {"audio_bytes": len(raw), "audio_ms": int(wav.size * 1000 / SR), "preprocess_ms": pre_ms, "asr_ms": asr_ms, "align_ms": align_ms}}

@app.websocket("/stream")
async def stream_ws(ws: WebSocket):
    token = ws.query_params.get("token") or ws.headers.get("x-api-token")
    if not token or not _secrets.compare_digest(token, API_TOKEN):
        await ws.close(code=1008, reason="invalid token")
        return
    await ws.accept()
    try:
        while True:
            ctrl = json.loads(await ws.receive_text())
            mode = str(ctrl.get("mode", "word")).lower()
            target = str(ctrl.get("ayah" if mode == "ayah" else "target", "")).strip()
            msg = await ws.receive()
            raw = msg.get("bytes")
            if not raw or not target:
                await ws.send_json({"error": "Zieltext oder Audiodaten fehlen."})
                continue
            if mode == "ayah":
                for frame in await asyncio.to_thread(lambda: list(_score_ayah_streamed(raw, target))):
                    await ws.send_json(frame)
                    if frame.get("kind") == "word":
                        await asyncio.sleep(WORD_STREAM_DELAY_SEC)
            else:
                await ws.send_json(await asyncio.to_thread(_score_word, raw, target))
    except WebSocketDisconnect:
        _log("ws_close", reason="disconnect")

print("API bereit: /health, /logs, /assess und /stream")

## Start des CPU-Backends und Tailscale-Funnels

Die Startzelle lädt keinen weiteren ASR-Stack. Sie startet nur den bereits definierten FastAPI-Prozess und veröffentlicht ihn über Tailscale Funnel. `TS_AUTHKEY` und optional `API_TOKEN` werden ausschließlich aus Secret/Umgebung gelesen.

In [ ]:
# ---- FastAPI + Tailscale Funnel -----------------------------------------
# Idempotent: einen vorhandenen tailscaled-Dienst weiterverwenden und nicht
# auf dem Laptop ungefragt beenden. TS_AUTHKEY aus Colab-Secret oder Umgebung.
import json, os, shutil, subprocess, sys, threading, time as _t, urllib.request
import uvicorn

PORT = 8000
IS_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")

_key = os.environ.get("TS_AUTHKEY", "").strip()
if not _key and IS_COLAB:
    try:
        from google.colab import userdata
        _key = (userdata.get("TS_AUTHKEY") or "").strip()
    except Exception:
        pass
if not _key:
    raise RuntimeError(
        "TS_AUTHKEY fehlt. In Colab als Secret TS_AUTHKEY setzen oder lokal "
        "als Umgebungsvariable exportieren; der Key wird nicht im Notebook gespeichert."
    )

if shutil.which("tailscale") is None:
    if IS_COLAB:
        print("📦 Installiere Tailscale …")
        subprocess.run(
            "curl -fsSL https://tailscale.com/install.sh | sh",
            shell=True, check=True, stdout=subprocess.DEVNULL,
            stderr=subprocess.STDOUT,
        )
    else:
        raise RuntimeError("tailscale fehlt. Tailscale installieren und Run All erneut starten.")

# Auf dem Laptop/bei einem bereits laufenden Colab-Dienst nichts killen.
def _tailscale_status():
    try:
        return json.loads(subprocess.check_output(
            ["tailscale", "status", "--json"], stderr=subprocess.STDOUT, timeout=10
        ))
    except Exception:
        return None

status_json = _tailscale_status()
if status_json is None and IS_COLAB:
    os.makedirs("/var/run/tailscale", exist_ok=True)
    os.makedirs("/var/lib/tailscale", exist_ok=True)
    subprocess.Popen(
        ["tailscaled", "--tun=userspace-networking",
         "--socks5-server=localhost:1055",
         "--state=/var/lib/tailscale/tailscaled.state",
         "--socket=/var/run/tailscale/tailscaled.sock"],
        stdout=open("/tmp/tailscaled.log", "a"), stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        if os.path.exists("/var/run/tailscale/tailscaled.sock"):
            break
        _t.sleep(0.5)
    else:
        raise RuntimeError("tailscaled-Socket nicht erreichbar; /tmp/tailscaled.log pruefen.")

# Auth-Key wird nur als Prozessargument an tailscale uebergeben und nie ausgegeben.
r = subprocess.run(
    ["tailscale", "up", f"--auth-key={_key}",
     "--hostname=colab-asr", "--accept-routes=false", "--timeout=30s"],
    capture_output=True, text=True, timeout=60,
)
if r.returncode != 0:
    raise RuntimeError(f"tailscale up fehlgeschlagen: {(r.stderr or r.stdout).strip()[:500]}")

_self = {}
for _ in range(60):
    status_json = _tailscale_status() or {}
    _self = status_json.get("Self") or {}
    if _self.get("Online") is True:
        break
    _t.sleep(1)
else:
    raise RuntimeError(
        f"Tailscale ist nach 60s offline: hostname={_self.get('HostName')} "
        f"online={_self.get('Online')}"
    )
print(f"✅ Tailscale online: {_self.get('HostName')} ({_self.get('DNSName', '').rstrip('.')})")

# FastAPI nur starten, wenn Port 8000 noch keinen gesunden Server hat.
def _fastapi_up():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if not _fastapi_up():
    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT,
                                   log_level="warning", access_log=False),
        daemon=True,
    ).start()
    for _ in range(30):
        if _fastapi_up():
            break
        _t.sleep(0.5)
    else:
        raise RuntimeError("FastAPI konnte auf Port 8000 nicht gestartet werden.")
print(f"✅ FastAPI bereit auf Port {PORT}")

DNS_NAME = (_self.get("DNSName") or "").rstrip(".")
if not DNS_NAME:
    raise RuntimeError("Tailscale-DNSName konnte nicht ermittelt werden.")
public_url = f"https://{DNS_NAME}"

# Funnel ist idempotent; bestehende Freigabe wird wiederverwendet.
funnel = subprocess.run(
    ["tailscale", "funnel", "--bg", str(PORT)],
    capture_output=True, text=True, timeout=120,
)
status = subprocess.run(
    ["tailscale", "funnel", "status"], capture_output=True, text=True, timeout=10,
)
status_text = (status.stdout or "") + (status.stderr or "")
if funnel.returncode != 0 and f":{PORT}" not in status_text and f"127.0.0.1:{PORT}" not in status_text:
    raise RuntimeError(
        "Tailscale Funnel konnte nicht aktiviert werden. "
        "HTTPS/Funnel-ACL im Tailscale-Admin pruefen.\n"
        + (funnel.stderr or funnel.stdout).strip()[:800]
    )
print("✅ Tailscale Funnel aktiv")

print("\n" + "=" * 68)
print(f"🌍 Backend-URL für die App:  {public_url}")
print(f"🔐 API-Token (in App eintragen):  {API_TOKEN}")
print("=" * 68)
print(f"Health-Check:  {public_url}/health")
print(f"WebSocket:     {public_url.replace('https://', 'wss://')}/stream?token={API_TOKEN}")
print(f"Live-Logs:     {public_url}/logs?token={API_TOKEN}")

## 🔎 Backend-Logs anschauen

Diese Zelle jederzeit **erneut ausführen**, um die letzten Backend-Timings zu sehen.  
Alternativ im Handy-Browser: `{PUBLIC_URL}/logs`  (HTML mit Auto-Refresh).


In [ ]:
# --- Backend-Logs Live (jederzeit erneut ausfuehren) ---
import subprocess
out = subprocess.run(["tail", "-n", "40", "/content/backend.log"], capture_output=True, text=True)
print(out.stdout or "(noch keine Logs)")
if 'public_url' in dir():
    print(f"\n🌍 Live im Browser: {public_url}/logs")


## Für die Übernahme ins Backend

* **Frame-Raster unverändert.** Der IR-Export ändert den Conv-Stack nicht; die ~20 ms/Frame und `blank_id = config.pad_token_id` gelten weiter.
* **`forced_align` braucht weiter torch/torchaudio.** Nur der ASR-Modell-Forward läuft über OpenVINO.
* **CPU-Laufzeit:** Das Modell wird mit `PERFORMANCE_HINT=LATENCY`, einem Stream, begrenzten Threads und `CACHE_DIR` kompiliert.
* **Bewertung:** Preprocessing, GOP-/LLR-Scoring und Tajweed-Schwellen bleiben identisch zum bestehenden Backend.